In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import ttest_ind, f_oneway, pearsonr, spearmanr
import warnings
warnings.filterwarnings('ignore')

print("Statistical analysis starting...")

# ------------------------------------------------------------
# 1. Load data
# ------------------------------------------------------------
df = pd.read_csv('inflation_cleaned.csv')
print(f"Loaded {len(df)} observations from 'inflation_cleaned.csv'")

# ------------------------------------------------------------
# 2. Research questions
# ------------------------------------------------------------
print("""
Research questions:
1. Is there a significant difference in inflation rates between regions?
2. Is there a correlation between inflation and GDP growth?
3. Has global inflation changed significantly over time?
4. Do countries with different income levels have different inflation rates?
""")

# ------------------------------------------------------------
# 3. ANOVA: Inflation by Region
# ------------------------------------------------------------
print("ANOVA: Inflation by Region")

region_groups = [df[df['Region'] == region]['Inflation_Rate_Capped'].dropna()
                 for region in df['Region'].unique() if len(df[df['Region'] == region]) > 30]

f_stat, p_value = f_oneway(*region_groups)

print(f"One-way ANOVA: Inflation by Region")
print(f"  F-statistic: {f_stat:.4f}")
print(f"  P-value: {p_value:.6f}")

if p_value < 0.05:
    print("  Result: Statistically significant (p < 0.05)")
    print("  -> Inflation rates differ significantly across regions.")
else:
    print("  Result: Not statistically significant (p >= 0.05)")

# Post-hoc: Pairwise t-tests with Bonferroni correction
print("Pairwise t-tests (with Bonferroni correction):")
regions = [r for r in df['Region'].unique() if len(df[df['Region'] == r]) > 30]

significant_pairs = []
for i in range(len(regions)):
    for j in range(i+1, len(regions)):
        data1 = df[df['Region'] == regions[i]]['Inflation_Rate_Capped'].dropna()
        data2 = df[df['Region'] == regions[j]]['Inflation_Rate_Capped'].dropna()
        t_stat, p_val = ttest_ind(data1, data2)
        # Bonferroni correction
        if p_val < 0.05 / (len(regions) * (len(regions)-1) / 2):
            significant_pairs.append((regions[i], regions[j], p_val))

if significant_pairs:
    print("  Significant pairs (after Bonferroni correction):")
    for pair in significant_pairs:
        print(f"  - {pair[0]} vs {pair[1]}: p = {pair[2]:.6f}")
else:
    print("  No significant pairwise differences after correction.")

# ------------------------------------------------------------
# 4. Correlation: Inflation vs GDP Growth
# ------------------------------------------------------------
print("\nCorrelation: Inflation vs GDP Growth")

corr_data = df[['Inflation_Rate_Capped', 'GDP_Growth']].dropna()

pearson_r, pearson_p = pearsonr(corr_data['Inflation_Rate_Capped'], corr_data['GDP_Growth'])
spearman_r, spearman_p = spearmanr(corr_data['Inflation_Rate_Capped'], corr_data['GDP_Growth'])

print(f"Pearson Correlation:")
print(f"  Coefficient: {pearson_r:.4f}")
print(f"  P-value: {pearson_p:.6f}")

print(f"Spearman Correlation:")
print(f"  Coefficient: {spearman_r:.4f}")
print(f"  P-value: {spearman_p:.6f}")

if abs(pearson_r) > 0.3 and pearson_p < 0.05:
    print("  Weak-to-moderate correlation detected.")
    if pearson_r < 0:
        print("  Inflation and GDP growth tend to move in opposite directions.")
    else:
        print("  Inflation and GDP growth tend to move in the same direction.")
else:
    print("  No significant linear correlation detected.")

# ------------------------------------------------------------
# 5. Time series: Has inflation changed over time?
# ------------------------------------------------------------
print("\nTime Series Analysis")

df_early = df[df['Year'] < 2008]
df_late = df[df['Year'] >= 2010]

early_inflation = df_early['Inflation_Rate_Capped'].dropna()
late_inflation = df_late['Inflation_Rate_Capped'].dropna()

t_stat, p_value = ttest_ind(early_inflation, late_inflation)

print(f"Comparing Inflation: Pre-2008 vs Post-2010")
print(f"  Pre-2008 mean: {early_inflation.mean():.2f}%")
print(f"  Post-2010 mean: {late_inflation.mean():.2f}%")
print(f"  T-statistic: {t_stat:.4f}")
print(f"  P-value: {p_value:.6f}")

if p_value < 0.05:
    print("  Result: Statistically significant difference (p < 0.05)")
    print("  -> Global inflation patterns have changed significantly.")
else:
    print("  Result: No significant difference detected.")

# ------------------------------------------------------------
# 6. Inflation by Income Level
# ------------------------------------------------------------
print("\nInflation by Income Level")

income_groups = df.groupby('IncomeLevel')['Inflation_Rate_Capped'].mean().sort_values(ascending=False)
print("Average Inflation by Income Level:")
print(income_groups.round(2))

income_levels = [g for g in df['IncomeLevel'].unique() if g != 'Unknown']
income_data = [df[df['IncomeLevel'] == level]['Inflation_Rate_Capped'].dropna()
               for level in income_levels if len(df[df['IncomeLevel'] == level]) > 30]

if len(income_data) >= 2:
    f_stat, p_value = f_oneway(*income_data)
    print(f"ANOVA: Inflation by Income Level")
    print(f"  F-statistic: {f_stat:.4f}")
    print(f"  P-value: {p_value:.6f}")
    if p_value < 0.05:
        print("  Result: Statistically significant (p < 0.05)")
        print("  -> Inflation varies significantly by income level.")
    else:
        print("  Result: Not statistically significant.")
else:
    print("  Not enough income groups with sufficient data for ANOVA.")

# ------------------------------------------------------------
# 7. Summary of findings
# ------------------------------------------------------------
print("\nSummary of Statistical Findings")

# Re-evaluate some significance flags from earlier sections
# (these variables are still in scope from the last ANOVA test; careful to not overwrite)
# We'll reassign them here using the last p_value from ANOVA income or region; actually need proper flags.
# In the original script they reused p_value, causing confusion. Better to compute fresh.
region_f_stat, region_p = f_oneway(*[df[df['Region'] == r]['Inflation_Rate_Capped'].dropna() 
                                     for r in regions if len(df[df['Region'] == r]) > 30])
corr_p = pearson_p  # from earlier
time_p = ttest_ind(early_inflation, late_inflation)[1]
income_p = None
if len(income_data) >= 2:
    income_p = f_oneway(*income_data)[1]
else:
    income_p = 1.0

print("Key Statistical Findings:")
print(f"1. Regional Differences: {'Significant' if region_p < 0.05 else 'Not significant'} (p={region_p:.4f})")
print(f"2. Inflation-GDP Relationship: {'Significant correlation' if pearson_p < 0.05 and abs(pearson_r) > 0.1 else 'No significant correlation'} (r={pearson_r:.2f}, p={pearson_p:.4f})")
print(f"3. Temporal Changes: {'Significant change' if time_p < 0.05 else 'No significant change'} (p={time_p:.4f})")
print(f"4. Income Level Impact: {'Significant' if income_p is not None and income_p < 0.05 else 'Not significant'} (p={income_p:.4f})" if income_p is not None else "4. Income Level Impact: Not enough data")

print("Statistical analysis complete.")